# Bias Transformer Regression - Colab

정치 편향도 `label1`을 1.0~5.0 연속 점수로 예측하는 Transformer 회귀 노트북입니다.

- 기본 모델은 `klue/roberta-base`입니다.
- 결과는 `metrics.json`과 `predictions.csv`로 저장합니다.
- split 생성과 학습을 분리했습니다.


In [ ]:
!pip -q install transformers datasets accelerate sentencepiece evaluate scikit-learn pandas joblib


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
import inspect
import time

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path('/content/drive/MyDrive/sw_project/ai_features/political_bias_analysis')
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'original_data'
SPLIT_DIR = DATA_DIR / 'splits_label1_regression'
MODEL_DIR = PROJECT_ROOT / 'regression/bias_transformer_regression/models'
OUTPUT_NAME = 'bias_transformer_regression'
MODEL_NAME = 'klue/roberta-base'
MAX_LENGTH = 512
BATCH_SIZE = 4
GRAD_ACCUM = 4
EPOCHS = 3

RAW_TRAIN = RAW_DIR / 'complete_train.csv'
RAW_TEST = RAW_DIR / 'complete_test.csv'
if not RAW_TRAIN.exists():
    RAW_TRAIN = RAW_DIR / 'complete_train_stratified.csv'
if not RAW_TEST.exists():
    RAW_TEST = RAW_DIR / 'complete_test_stratified.csv'

TRAIN_CSV = SPLIT_DIR / 'train.csv'
VALID_CSV = SPLIT_DIR / 'valid.csv'
TEST_CSV = SPLIT_DIR / 'test.csv'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print('project_root =', PROJECT_ROOT)
print('data_dir =', DATA_DIR)
print('raw_dir =', RAW_DIR)
print('split_dir =', SPLIT_DIR)
print('model_dir =', MODEL_DIR)
print('raw_train =', RAW_TRAIN)
print('raw_test =', RAW_TEST)


def compose_text(df):
    title = df['title'].fillna('').astype(str).str.strip()
    content = df['content'].fillna('').astype(str).str.strip()
    return (title + ' ' + content).str.strip()


def load_raw_frame(path):
    df = pd.read_csv(path)
    df = df.copy()
    df['text'] = compose_text(df)
    df['label'] = df['label1'].astype(float)
    cols = [c for c in ['seq', 'title', 'content', 'text', 'label'] if c in df.columns]
    return df[cols + [c for c in df.columns if c not in cols]]


In [ ]:
import subprocess


def _log(msg):
    print(f'[split] {msg}')


def ensure_splits():
    split_start = time.perf_counter()
    _log(f'train_source = {RAW_TRAIN.exists()} | {RAW_TRAIN}')
    _log(f'test_source = {RAW_TEST.exists()} | {RAW_TEST}')

    if TRAIN_CSV.exists() and VALID_CSV.exists() and TEST_CSV.exists():
        _log('split files already exist; skipping creation')
        print(f'[split] elapsed = {time.perf_counter() - split_start:.2f}s')
        return

    if not RAW_TRAIN.exists() or not RAW_TEST.exists():
        raise FileNotFoundError('Raw CSV files are missing. Need original_data/complete_train.csv and complete_test.csv.')

    result = subprocess.run(
        [
            'python',
            str(PROJECT_ROOT / 'prepare_label1_splits.py'),
            '--label-mode', 'five_class',
            '--train-csv', str(RAW_TRAIN),
            '--test-csv', str(RAW_TEST),
            '--out-dir', str(SPLIT_DIR),
        ],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    print(result.stderr)
    result.check_returncode()
    _log('split generation completed')
    print(f'[split] elapsed = {time.perf_counter() - split_start:.2f}s')


ensure_splits()


In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)


def _log(msg):
    print(f'[train] {msg}')


def load_split(path):
    df = pd.read_csv(path)
    if 'text' not in df.columns:
        title = df['title'].fillna('').astype(str).str.strip()
        content = df['content'].fillna('').astype(str).str.strip()
        df['text'] = (title + ' ' + content).str.strip()
    df['label'] = df['label'].astype(float)
    return df[['text', 'label']].copy(), df


def make_dataset(df):
    ds = Dataset.from_pandas(df, preserve_index=False)
    return ds


def tokenize_function(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


def regression_metrics_from_preds(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    pearson = pd.Series(y_true).corr(pd.Series(y_pred), method='pearson')
    spearman = pd.Series(y_true).corr(pd.Series(y_pred), method='spearman')
    rounded_true = np.clip(np.rint(y_true), 1, 5).astype(int)
    rounded_pred = np.clip(np.rint(y_pred), 1, 5).astype(int)
    acc = accuracy_score(rounded_true, rounded_pred)
    rounded_macro_f1 = f1_score(rounded_true, rounded_pred, average='macro')
    return {
        'mae': float(mae),
        'rmse': float(rmse),
        'pearson': float(pearson),
        'spearman': float(spearman),
        'rounded_accuracy': float(acc),
        'rounded_macro_f1': float(rounded_macro_f1),
    }


from sklearn.metrics import f1_score

train_df, train_raw = load_split(TRAIN_CSV)
valid_df, valid_raw = load_split(VALID_CSV)
test_df, test_raw = load_split(TEST_CSV)

raw_dataset = DatasetDict({
    'train': make_dataset(train_df),
    'validation': make_dataset(valid_df),
    'test': make_dataset(test_df),
})

_log('loading tokenizer/model')
model_load_start = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    problem_type='regression',
    id2label={0: 'score'},
    label2id={'score': 0},
)
_log(f'model loaded in {time.perf_counter() - model_load_start:.2f}s')

encoded = raw_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
encoded = encoded.rename_column('label', 'labels')
encoded.set_format(type='torch')

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


def build_training_args():
    base_kwargs = {
        'output_dir': str(MODEL_DIR / 'checkpoints'),
        'learning_rate': 2e-5,
        'per_device_train_batch_size': BATCH_SIZE,
        'per_device_eval_batch_size': BATCH_SIZE,
        'gradient_accumulation_steps': GRAD_ACCUM,
        'num_train_epochs': EPOCHS,
        'weight_decay': 0.01,
        'logging_steps': 10,
        'save_strategy': 'epoch',
        'save_total_limit': 1,
        'load_best_model_at_end': True,
        'metric_for_best_model': 'rmse',
        'greater_is_better': False,
        'report_to': 'none',
        'fp16': torch.cuda.is_available(),
        'dataloader_num_workers': 2,
    }
    sig = inspect.signature(TrainingArguments)
    params = sig.parameters
    if 'eval_strategy' in params:
        base_kwargs['eval_strategy'] = 'epoch'
    elif 'evaluation_strategy' in params:
        base_kwargs['evaluation_strategy'] = 'epoch'
    return TrainingArguments(**{k: v for k, v in base_kwargs.items() if k in params})


def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.squeeze(preds)
    labels = np.squeeze(labels)
    return regression_metrics_from_preds(labels, preds)


training_args = build_training_args()
trainer_sig = inspect.signature(Trainer)
trainer_kwargs = {
    'model': model,
    'args': training_args,
    'train_dataset': encoded['train'],
    'eval_dataset': encoded['validation'],
    'data_collator': data_collator,
    'compute_metrics': compute_metrics,
    'callbacks': [EarlyStoppingCallback(early_stopping_patience=1)],
}
if 'tokenizer' in trainer_sig.parameters:
    trainer_kwargs['tokenizer'] = tokenizer
trainer = Trainer(**trainer_kwargs)

_log('training starts')
train_start = time.perf_counter()
trainer.train()
_log(f'training completed in {time.perf_counter() - train_start:.2f}s')

_log('evaluating validation/test')
valid_pred = trainer.predict(encoded['validation'])
test_pred = trainer.predict(encoded['test'])

valid_scores = np.squeeze(valid_pred.predictions)
valid_labels = np.squeeze(valid_pred.label_ids)
test_scores = np.squeeze(test_pred.predictions)
test_labels = np.squeeze(test_pred.label_ids)

valid_metrics = regression_metrics_from_preds(valid_labels, valid_scores)
test_metrics = regression_metrics_from_preds(test_labels, test_scores)

print('[validation]', valid_metrics)
print('[test]', test_metrics)

metrics = {
    'model_name': MODEL_NAME,
    'split_name': 'splits_label1_regression',
    'max_length': MAX_LENGTH,
    'batch_size': BATCH_SIZE,
    'grad_accum': GRAD_ACCUM,
    'valid': valid_metrics,
    'test': test_metrics,
}

metrics_path = MODEL_DIR / f'{OUTPUT_NAME}_metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
print('saved metrics to', metrics_path)

pred_df = test_raw.copy()
pred_df['pred_score'] = test_scores
pred_df['rounded_pred_label'] = np.clip(np.rint(test_scores), 1, 5).astype(int)
pred_df.to_csv(MODEL_DIR / f'{OUTPUT_NAME}_predictions.csv', index=False)
print('saved predictions to', MODEL_DIR / f'{OUTPUT_NAME}_predictions.csv')

trainer.save_model(MODEL_DIR / OUTPUT_NAME)
tokenizer.save_pretrained(MODEL_DIR / OUTPUT_NAME)
print('saved model to', MODEL_DIR / OUTPUT_NAME)


In [ ]:
def predict_score(title, content):
    text = f'{title.strip()} {content.strip()}'.strip()
    encoded = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_LENGTH,
    )
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    trainer.model.to(device)
    encoded = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        score = trainer.model(**encoded).logits.squeeze().item()
    clipped = float(np.clip(score, 1.0, 5.0))
    rounded = int(np.clip(np.rint(clipped), 1, 5))
    return {
        'pred_score': clipped,
        'rounded_label': rounded,
    }

# 예시
# predict_score('기사 제목', '기사 본문')
